# 🔗 Day 04a: Pandas Merge & Join — SQL JOINs in Python

---

## 🎯 What You'll Master Today
- merge() with all join types (inner, left, right, outer)
- join() for index-based joins
- concat() for stacking DataFrames
- Indicator column for debugging joins
- Handling duplicates in merges

**Goal:** Replicate any SQL JOIN in Pandas from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  JOIN TYPES                                                       ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  INNER:  Only matching rows from both tables   [A ∩ B]         ║
║  LEFT:   All from left + matching from right   [A + A∩B]       ║
║  RIGHT:  All from right + matching from left   [B + A∩B]       ║
║  OUTER:  All rows from both tables             [A ∪ B]         ║
║                                                                   ║
║  merge()  → column-based (like SQL JOIN ON)                    ║
║  join()   → index-based (convenience for index joins)          ║
║  concat() → stacking (UNION ALL / row/col append)              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import pandas as pd
import numpy as np

# Setup: Two related tables
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'dept_id': [101, 102, 101, 103, 104]  # 104 doesn't exist in depts
})

departments = pd.DataFrame({
    'dept_id': [101, 102, 103, 105],  # 105 has no employees
    'dept_name': ['Engineering', 'Sales', 'HR', 'Marketing']
})

print(f"Employees:\n{employees}\n")
print(f"Departments:\n{departments}")

In [ ]:
# ============================================================
# 1. merge() — The 4 Join Types
# ============================================================

# INNER JOIN (default) — only matching rows
inner = pd.merge(employees, departments, on='dept_id', how='inner')
print(f"INNER JOIN:\n{inner}\n")
# Eve (dept 104) dropped, Marketing (105) dropped

# LEFT JOIN — keep all from left
left = pd.merge(employees, departments, on='dept_id', how='left')
print(f"LEFT JOIN:\n{left}\n")
# Eve kept with NaN dept_name

# RIGHT JOIN — keep all from right
right = pd.merge(employees, departments, on='dept_id', how='right')
print(f"RIGHT JOIN:\n{right}\n")
# Marketing kept with NaN employee

# OUTER JOIN — keep everything
outer = pd.merge(employees, departments, on='dept_id', how='outer')
print(f"OUTER JOIN:\n{outer}")

In [ ]:
# ============================================================
# 2. Merge on Different Column Names
# ============================================================

# When columns have different names
orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4],
    'customer': [1, 2, 1, 3],
    'amount': [100, 200, 150, 300]
})

customers = pd.DataFrame({
    'cust_id': [1, 2, 4],
    'cust_name': ['Alice', 'Bob', 'Diana']
})

# left_on / right_on for different column names
result = pd.merge(orders, customers, left_on='customer', right_on='cust_id', how='left')
print(f"Different column names:\n{result}\n")

# Merge on multiple columns
sales_q1 = pd.DataFrame({'dept': ['Eng', 'Sales'], 'quarter': ['Q1', 'Q1'], 'rev': [100, 200]})
sales_q2 = pd.DataFrame({'dept': ['Eng', 'Sales'], 'quarter': ['Q2', 'Q2'], 'rev': [110, 180]})
all_sales = pd.concat([sales_q1, sales_q2]).reset_index(drop=True)
targets = pd.DataFrame({'dept': ['Eng', 'Eng', 'Sales', 'Sales'], 
                         'quarter': ['Q1', 'Q2', 'Q1', 'Q2'],
                         'target': [120, 100, 180, 190]})
result = pd.merge(all_sales, targets, on=['dept', 'quarter'])
print(f"Multi-column merge:\n{result}")

In [ ]:
# ============================================================
# 3. Indicator Column — Debug Your Joins
# ============================================================

# indicator=True adds _merge column showing where each row came from
result = pd.merge(employees, departments, on='dept_id', how='outer', indicator=True)
print(f"With indicator:\n{result}\n")

# Find unmatched rows
left_only = result[result['_merge'] == 'left_only']
print(f"Employees without dept:\n{left_only[['name', 'dept_id']]}\n")

right_only = result[result['_merge'] == 'right_only']
print(f"Depts without employees:\n{right_only[['dept_id', 'dept_name']]}")

print("\n💡 indicator=True is your best debugging tool for joins!")

In [ ]:
# ============================================================
# 4. Handling Duplicates in Joins
# ============================================================

# Many-to-many (cartesian product of matches)
left = pd.DataFrame({'key': ['A', 'A', 'B'], 'val_left': [1, 2, 3]})
right = pd.DataFrame({'key': ['A', 'A', 'B'], 'val_right': [10, 20, 30]})

result = pd.merge(left, right, on='key')
print(f"Many-to-many (watch out!):\n{result}\n")
# A×A gives 4 rows! (2 left × 2 right)

# validate= parameter to catch unexpected duplicates
try:
    pd.merge(left, right, on='key', validate='one_to_one')
except Exception as e:
    print(f"Validation error: {e}")

# Suffixes for overlapping column names
df1 = pd.DataFrame({'id': [1, 2], 'value': [10, 20]})
df2 = pd.DataFrame({'id': [1, 2], 'value': [100, 200]})
result = pd.merge(df1, df2, on='id', suffixes=('_left', '_right'))
print(f"\nWith suffixes:\n{result}")

In [ ]:
# ============================================================
# 5. concat() — Stacking DataFrames
# ============================================================

df1 = pd.DataFrame({'name': ['Alice', 'Bob'], 'score': [90, 85]})
df2 = pd.DataFrame({'name': ['Charlie', 'Diana'], 'score': [88, 92]})
df3 = pd.DataFrame({'name': ['Eve'], 'score': [95]})

# Vertical stacking (like SQL UNION ALL)
combined = pd.concat([df1, df2, df3], ignore_index=True)
print(f"Vertical concat:\n{combined}\n")

# Horizontal stacking (add columns side-by-side)
info = pd.DataFrame({'age': [25, 30], 'city': ['NYC', 'LA']})
combined_h = pd.concat([df1, info], axis=1)
print(f"Horizontal concat:\n{combined_h}\n")

# With different columns (outer join by default)
df_a = pd.DataFrame({'name': ['Alice'], 'score': [90], 'grade': ['A']})
df_b = pd.DataFrame({'name': ['Bob'], 'score': [85], 'rank': [2]})
combined = pd.concat([df_a, df_b], ignore_index=True)
print(f"Different columns (fills NaN):\n{combined}")

In [ ]:
# ============================================================
# 6. join() — Index-Based Joining
# ============================================================

# join() is a shortcut for index-based merging
emp = employees.set_index('dept_id')
dept = departments.set_index('dept_id')

# Left join on index
result = emp.join(dept, how='left')
print(f"join() on index:\n{result}\n")

# Equivalent merge:
result2 = pd.merge(employees, departments, on='dept_id', how='left')
print(f"Same with merge:\n{result2}")

print("\n💡 merge() is more flexible. join() is convenient when index is the key.")

In [ ]:
# ============================================================
# 7. Complex Merge Scenarios
# ============================================================

# Scenario 1: Self-join (manager hierarchy)
emp_mgr = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'manager_id': [None, 1, 1, 2]
})
result = pd.merge(
    emp_mgr, emp_mgr[['emp_id', 'name']],
    left_on='manager_id', right_on='emp_id',
    how='left', suffixes=('', '_mgr')
)
print(f"Self-join (managers):\n{result[['name', 'name_mgr']]}\n")

# Scenario 2: Anti-join (rows in A but NOT in B)
# Find employees not in any department
merged = pd.merge(employees, departments, on='dept_id', how='left', indicator=True)
anti_join = merged[merged['_merge'] == 'left_only']
print(f"Anti-join (no dept):\n{anti_join[['name', 'dept_id']]}\n")

# Scenario 3: Cross join (cartesian product)
colors = pd.DataFrame({'color': ['red', 'blue']})
sizes = pd.DataFrame({'size': ['S', 'M', 'L']})
cross = pd.merge(colors, sizes, how='cross')
print(f"Cross join:\n{cross}")

---

## 🗺️ Merge & Join Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  MERGE & JOIN — SQL to Pandas                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  SQL JOIN ON        → pd.merge(df1, df2, on='col')              ║
║  LEFT/RIGHT/OUTER   → how='left'/'right'/'outer'               ║
║  Join diff cols     → left_on='a', right_on='b'                ║
║  UNION ALL          → pd.concat([df1, df2], ignore_index=True) ║
║  Anti-join          → merge(indicator=True) + filter left_only ║
║  Cross join         → merge(how='cross')                       ║
║  Self join          → merge(df, df, left_on, right_on)         ║
║                                                                   ║
║  DEBUG:  indicator=True → shows 'both/left_only/right_only'   ║
║  GUARD:  validate='one_to_one' catches unexpected duplicates   ║
║  NAMES:  suffixes=('_x', '_y') for overlapping columns         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | merge() vs join() vs concat()? | merge = column-based SQL JOINs. join = index-based. concat = stacking (UNION ALL) |
| 2 | How to do an anti-join? | merge with indicator=True, then filter _merge == 'left_only' |
| 3 | What happens with duplicate keys? | Many-to-many creates cartesian product. Use validate= to catch this |
| 4 | How to debug a bad join? | indicator=True, check shape before/after, validate= parameter |
| 5 | merge on vs left_on/right_on? | on= when same column name in both. left_on/right_on when different names |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • merge() = SQL JOINs (inner/left/right/outer)         │
## │  • concat() = stacking (UNION ALL)                      │
## │  • indicator=True for debugging joins                   │
## │  • validate= to catch duplicate key issues              │
## │  • Anti-join = left merge + filter left_only            │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Data Cleaning** — Missing values, duplicates, type conversion, pipe()